# TA Session Analyzer — raw Colab (no Streamlit)

Runs the analysis pipeline directly as Python — upload a recording (or paste a URL), it prints the scorecard and gives you PDF/JSON downloads.

You'll need two free API keys:
- **Gemini** (analysis/scoring): https://aistudio.google.com/app/apikey
- **Deepgram** (transcription/diarization, free credits): https://console.deepgram.com

Run the cells top to bottom.

In [ ]:
#@title 1. Clone the repo
BRANCH = "claude/ta-session-analyzer-z2n1ml"  #@param {type:"string"}

import os
if os.path.exists("/content/ta-session-analyzer"):
    %cd /content/ta-session-analyzer
    !git fetch origin $BRANCH
    !git checkout $BRANCH
    !git pull origin $BRANCH
else:
    %cd /content
    !git clone -b $BRANCH https://github.com/MAHI-0828/ta-session-analyzer.git
    %cd ta-session-analyzer

In [ ]:
#@title 2. Install dependencies
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null
!pip install -q -r requirements.txt

In [ ]:
#@title 3. Enter your API keys (hidden input, not saved to disk)
from getpass import getpass

GEMINI_API_KEY = getpass("Gemini API key: ")
DEEPGRAM_API_KEY = getpass("Deepgram API key: ")
print("Keys captured for this session.")

In [ ]:
#@title 4. Get the recording — upload a file OR paste a URL
#@markdown Leave RECORDING_URL blank to upload a file from your computer instead.
RECORDING_URL = ""  #@param {type:"string"}

import os
from recording_utils import extract_video_url, download_video

video_path = "/content/session_recording.mp4"

if RECORDING_URL.strip():
    resolved = extract_video_url(RECORDING_URL.strip())
    print(f"Downloading from: {resolved}")
    download_video(resolved, video_path)
else:
    from google.colab import files
    print("Choose a video file to upload...")
    uploaded = files.upload()
    src_name = next(iter(uploaded))
    os.rename(src_name, video_path)

print(f"Video ready at {video_path} ({os.path.getsize(video_path) / 1e6:.1f} MB)")

In [ ]:
#@title 5. (Optional) Session metadata + chat log
SESSION_ID = "session_001"  #@param {type:"string"}
TA_NAME = ""  #@param {type:"string"}
STUDENT_NAME = ""  #@param {type:"string"}
ANALYZE_SCREEN = True  #@param {type:"boolean"}

#@markdown Optional: upload a plain-text chat log to also flag pasted solutions in chat.
UPLOAD_CHAT_LOG = False  #@param {type:"boolean"}

chat_text = None
if UPLOAD_CHAT_LOG:
    from google.colab import files
    uploaded = files.upload()
    chat_path = next(iter(uploaded))
    with open(chat_path, encoding="utf-8") as f:
        chat_text = f.read()
    print(f"Loaded chat log: {chat_path}")

In [ ]:
#@title 6. Run the analysis
from ta_core import analyze_ta_session

report = analyze_ta_session(
    GEMINI_API_KEY, video_path, DEEPGRAM_API_KEY,
    analyze_screen=ANALYZE_SCREEN, chat_text=chat_text,
)

breakdown = report["score_breakdown"]
analysis = report["analysis"]

print(f"Overall Score: {breakdown['overall']:.1f} / 100")
print(f"Doubt Resolution: {analysis['doubt_resolution']['status']}")
print(f"Duration: {report['duration_minutes']} min")
print(f"Transcription confidence: {report['transcription_confidence_pct']}%")
print()

p = report["participation"]
print(f"Participation — TA {p['ta_pct']}% | Student {p['student_pct']}%")
print()

print("Score Breakdown:")
for key, val in breakdown.items():
    if key == "overall":
        continue
    print(f"  {key.replace('_', ' ').title():<22} {val['points']:>5} / {val['max']}")
print()

if report["flags"]:
    print("AI Flags — recommended for manual review:")
    for flag in report["flags"]:
        print(f"  - {flag}")
else:
    print("No flags — session looks healthy.")
print()

print("Summary:")
print(analysis.get("summary", ""))
print()

if analysis.get("recommendations"):
    print("Recommendations:")
    for rec in analysis["recommendations"]:
        print(f"  - {rec}")

In [ ]:
#@title 7. Print the full transcript

In [ ]:
print(report["transcript_text"])

In [ ]:
#@title 8. Save + download PDF and JSON reports
import json
from ta_pdf_report import generate_ta_pdf
from google.colab import files

session_meta = {"session_id": SESSION_ID, "ta_name": TA_NAME, "student_name": STUDENT_NAME}

pdf_bytes = generate_ta_pdf(session_meta, report)
pdf_path = f"/content/{SESSION_ID}.pdf"
with open(pdf_path, "wb") as f:
    f.write(pdf_bytes)

json_path = f"/content/{SESSION_ID}.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump({"session_meta": session_meta, "report": report}, f, indent=2, ensure_ascii=False)

print(f"Saved {pdf_path} and {json_path}")
files.download(pdf_path)
files.download(json_path)

---
## Bulk: analyze multiple videos at once

Upload several recordings directly (no CSV or URLs needed) and get a PDF + JSON per session plus one rollup CSV. Run cells 9-11 instead of 4-8 above.

In [ ]:
#@title 9. Bulk upload — select multiple video files at once
from google.colab import files
import os

BULK_DIR = "/content/bulk_videos"
os.makedirs(BULK_DIR, exist_ok=True)

print("Select one or more video files to upload (ctrl/cmd-click to multi-select)...")
uploaded = files.upload()

bulk_video_paths = []
for name in uploaded:
    dest = os.path.join(BULK_DIR, name)
    os.rename(name, dest)
    bulk_video_paths.append(dest)

print(f"\n{len(bulk_video_paths)} video(s) ready:")
for p in bulk_video_paths:
    print(f"  - {p}")

In [ ]:
#@title 10. Run the analysis on every uploaded video
ANALYZE_SCREEN_BULK = True  #@param {type:"boolean"}

import json, os, re, traceback
from ta_core import analyze_ta_session
from ta_pdf_report import generate_ta_pdf

BULK_OUTPUT_DIR = "/content/ta_bulk_reports"
os.makedirs(BULK_OUTPUT_DIR, exist_ok=True)

bulk_results = []
bulk_errors = []

for i, video_path in enumerate(bulk_video_paths, 1):
    session_id = re.sub(r"[^A-Za-z0-9_.-]", "-", os.path.splitext(os.path.basename(video_path))[0])
    print(f"\n[{i}/{len(bulk_video_paths)}] {session_id} — analyzing...")
    try:
        report = analyze_ta_session(
            GEMINI_API_KEY, video_path, DEEPGRAM_API_KEY,
            analyze_screen=ANALYZE_SCREEN_BULK, chat_text=None,
        )
        session_meta = {"session_id": session_id, "ta_name": "", "student_name": ""}

        pdf_bytes = generate_ta_pdf(session_meta, report)
        with open(os.path.join(BULK_OUTPUT_DIR, f"{session_id}.pdf"), "wb") as f:
            f.write(pdf_bytes)
        with open(os.path.join(BULK_OUTPUT_DIR, f"{session_id}.json"), "w", encoding="utf-8") as f:
            json.dump({"session_meta": session_meta, "report": report}, f, indent=2, ensure_ascii=False)

        overall = report["score_breakdown"]["overall"]
        print(f"  done — {overall}/100, flags: {report['flags'] or 'none'}")
        bulk_results.append({
            "session_id": session_id,
            "duration_minutes": report["duration_minutes"],
            "overall_score": overall,
            "doubt_resolution": report["analysis"]["doubt_resolution"]["status"],
            "ta_speaking_pct": report["participation"]["ta_pct"],
            "student_speaking_pct": report["participation"]["student_pct"],
            "flags": "; ".join(report["flags"]) or "none",
        })
    except Exception as e:
        print(f"  FAILED — {e}")
        traceback.print_exc()
        bulk_errors.append({"session_id": session_id, "error": str(e)})

print(f"\nDone. {len(bulk_results)} succeeded, {len(bulk_errors)} failed.")

In [ ]:
#@title 11. Download rollup CSV + all PDFs/JSON (zip)
import csv, zipfile
from google.colab import files

rollup_csv_path = os.path.join(BULK_OUTPUT_DIR, "rollup.csv")
with open(rollup_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["session_id", "duration_minutes", "overall_score",
                                            "doubt_resolution", "ta_speaking_pct",
                                            "student_speaking_pct", "flags"])
    writer.writeheader()
    writer.writerows(bulk_results)

if bulk_errors:
    print("Failures:")
    for e in bulk_errors:
        print(f"  - {e['session_id']}: {e['error']}")

zip_path = "/content/ta_bulk_reports.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for fname in os.listdir(BULK_OUTPUT_DIR):
        zf.write(os.path.join(BULK_OUTPUT_DIR, fname), arcname=fname)

print(f"\n{len(bulk_results)} report(s) zipped — downloading...")
files.download(zip_path)

---
### Alternative: run a whole day's CSV batch instead of one session

Upload `ta_sessions_today.csv` (columns: `recording_url,ta_name,student_name,session_id,analyze_screen,chat_log_path`) via the Colab file browser on the left, then run the cell below. Reports land in `ta_reports/`.

In [ ]:
#@title (Optional) Run the daily batch CLI instead
import os
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["DEEPGRAM_API_KEY"] = DEEPGRAM_API_KEY
!python ta_session_analyzer.py ta_sessions_today.csv